In [1]:
import logging

import pandas as pd

import data.helpers as dh
import models.helpers as mh
import cfr.cfr_helpers as cfrh
import cfr.cfr_viz_helpers as vh

import data.cfr_data_19_23 as cfrd
import data.breathe_data as bd
import datetime

# Load, process, save yearly data

In [ ]:
# Load 2019 data
df19 = cfrd.build_cfr_df(2019)
# WARNINGS:
# Surprisingly low pFEV1 are old females (69-80)
# Surprisingly high pFEV1 is a tall man (1.93m)

KeyboardInterrupt: 

In [ ]:
df19old = bd.load_meas_from_excel("CF_Registry_19_processed", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [ ]:
df19.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed.xlsx",
    index=False,
)

In [ ]:
df23 = cfrd.build_cfr_df(2023)

INFO:root:Loaded 10344 entries
INFO:root:5065 after removing all NaN
INFO:root:2701 entries after removing <18yr


In [ ]:
df23.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_23_processed.xlsx",
    index=False,
)

## Get best FEV1 (2012-2015, 2016-2019)

In [125]:
# df = cfrd.load_cfr_data(
#     2016, ["s01caseid_original", "s03clibestfev1"], ["ID", "best FEV1"]
# )

# years = [2016, 2017, 2018]
years = [2012, 2013, 2014, 2015]
df = pd.DataFrame(columns=["ID", "best FEV1", "Date Recorded"])

for year in years:
    dftmp = (
        pd.read_csv(
            dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
            usecols=["ID", "Best FEV1"],
            encoding="cp1252",
        )
        .dropna()
        .rename(columns={"Best FEV1": "best FEV1"})
    )
    dftmp["Date Recorded"] = pd.to_datetime(f"{year}-01-01").date()
    df = pd.concat([df, dftmp], ignore_index=True)

# Fixes
df.loc[df["best FEV1"] == "4..10", "best FEV1"] = 4.10
df.loc[df["best FEV1"] == "2..3", "best FEV1"] = 2.3
df.loc[df["best FEV1"] == "2..95", "best FEV1"] = 2.95
df.loc[df["best FEV1"] == "2..65", "best FEV1"] = 2.65
# B159409 has ..55. Corrected to 3.55 since it's also the best FEV1 for 2013
df.loc[(df.ID == "B159409") & (df["best FEV1"] == "..55"), "best FEV1"] = 3.55
df.loc[(df.ID == "B166656") & (df["best FEV1"] == "1.1.2"), "best FEV1"] = 1.12
# Drop values that can't be processed
idx = df.loc[df["best FEV1"] == "2/1/1958 0:00"].index
df = df.drop(idx).reset_index(drop=True)
idx = df.loc[df["best FEV1"] == "11/2/2013 0:00"].index
df = df.drop(idx).reset_index(drop=True)
idx = df.loc[df["best FEV1"] == "102%"].index
df = df.drop(idx).reset_index(drop=True)
idx = df.loc[df["best FEV1"] == "1/1/1941 0:00"].index
df = df.drop(idx).reset_index(drop=True)
idx = df.loc[df["best FEV1"] == "99%"].index
df = df.drop(idx).reset_index(drop=True)
idx = df.loc[df["best FEV1"] == "116%"].index
df = df.drop(idx).reset_index(drop=True)

df["best FEV1"] = df["best FEV1"].astype(float)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_55471/4036605979.py:11: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(
/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_55471/4036605979.py:11: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(
/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_55471/4036605979.py:11: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(


In [126]:
df = pd.concat([df, df19[["ID", "best FEV1", "Date Recorded"]]], ignore_index=True)

In [127]:
import data.sanity_checks as sc

# df.apply(lambda row: sc.fev1(row['best FEV1'], row['ID']), axis=1)
# ~35 individuals with FEV1 values as FEV1%Pred or really above 7! -> just drop them
df = df[(df["best FEV1"] < 7) & (df["best FEV1"] > 0.2)]
df.apply(lambda row: sc.fev1(row["best FEV1"], row["ID"]), axis=1)

0       -1
1       -1
2       -1
3       -1
4       -1
        ..
26358   -1
26359   -1
26360   -1
26361   -1
26362   -1
Length: 26337, dtype: int64

In [128]:
def get_back_bfev1(df_for_ID):
    """
    Find the max 'best FEV1' recorded during the past 4 annual reviews
    """
    # if not df_for_ID['Date Recorded'].isin([pd.to_datetime("2019-01-01").date()]).any():
    # return
    idx_max = df_for_ID["best FEV1"].idxmax()
    return {
        # "best FEV1 2016-19": df_for_ID.loc[idx_max, "best FEV1"],
        "best FEV1 2012-15": df_for_ID.loc[idx_max, "best FEV1"],
        "best FEV1 year": df_for_ID.loc[idx_max, "Date Recorded"],
    }


df_bfev1s = (
    df.groupby("ID").apply(lambda x: get_back_bfev1(x)).apply(pd.Series).reset_index()
)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_55471/2466117111.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("ID").apply(lambda x: get_back_bfev1(x)).apply(pd.Series).reset_index()


### Merge back columns into 2015 data

In [135]:
year = 2015
dftmp = (
    pd.read_csv(
        dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
        usecols=["ID", "Date of Birth", "Gender", "Height", "FEV1", "Best FEV1"],
        encoding="cp1252",
    )
    .dropna()
    .rename(columns={"Best FEV1": "best FEV1", "Gender": "Sex"})
)
dftmp["Date Recorded"] = pd.to_datetime(f"{year}-01-01").date()
dftmp["Date of Birth"] = pd.to_datetime(dftmp["Date of Birth"], errors="coerce")
dftmp["Age"] = dftmp["Date Recorded"].apply(lambda x: x.year) - dftmp[
    "Date of Birth"
].apply(lambda x: x.year)
dftmp.drop(columns=["Date of Birth"], inplace=True)
print(f"{dftmp[dftmp['Age'] >= 18].shape[0]} adults")
print(f"{dftmp[dftmp['Age'] < 18].shape[0]} children")
dftmp = dftmp[dftmp["Age"] >= 18]

4024 adults
2410 children


In [136]:
print(f"{df_bfev1s.shape[0]} patients with best FEV1 recorded between 2012 and 2015")
print(f"{dftmp.shape[0]} patients with data recorded in 2015")
dftmp = dftmp.merge(df_bfev1s, on=["ID"])
print(
    f"{dftmp.shape[0]} patients with data recorded in 2015 and best FEV1 recorded between 2012 and 2015"
)

7666 patients with best FEV1 recorded between 2012 and 2015
4024 patients with data recorded in 2015
4024 patients with data recorded in 2015 and best FEV1 recorded between 2012 and 2015


In [141]:
df19 = dftmp.copy()
col = "best FEV1 2012-15"
# col = "best FEV1 2016-19"
df19["bFEV1 diff"] = df19[col] - df19["best FEV1"]
df19["bFEV1 % diff"] = (df19[col] - df19["best FEV1"]) / df19["best FEV1"] * 100
df19 = df19.sort_values(by="bFEV1 % diff", ascending=False)

for i in [100, 50, 20, 10, 5]:
    n = df19[df19["bFEV1 % diff"] > i].shape[0]
    print(
        f"{n} ({n/df19.shape[0]:.2%}) entries with older bFEV1 {i}% over the value from 2019",
    )

49 (1.22%) entries with older bFEV1 100% over the value from 2019
236 (5.86%) entries with older bFEV1 50% over the value from 2019
765 (19.01%) entries with older bFEV1 20% over the value from 2019
1543 (38.34%) entries with older bFEV1 10% over the value from 2019
2214 (55.02%) entries with older bFEV1 5% over the value from 2019


In [142]:
dftmp.head()

,ID,Sex,Height,FEV1,best FEV1,Date Recorded,Age,best FEV1 2012-15,best FEV1 year
0,B155916,Female,164.0,1.56,1.84,2015-01-01,28,1.99,2014-01-01
1,B155917,Male,175.0,2.91,3.05,2015-01-01,42,3.47,2012-01-01
2,B155918,Male,191.0,4.57,4.95,2015-01-01,30,5.20,2013-01-01
3,B155920,Male,171.0,4.12,4.56,2015-01-01,35,4.82,2012-01-01
4,B155921,Female,150.0,1.23,1.42,2015-01-01,30,1.51,2012-01-01


In [ ]:
dftmp.to_excel(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/CF_Registry_15_processed_with_bFEV1_2012-15.xlsx",
    index=False,
)

# Load data

## Link 2019 with 2023 data

In [16]:
df19 = bd.load_meas_from_excel("CF_Registry_19_processed_with_idx", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [18]:
df19.head()

,ID,Age,Height,FEV1,FEF2575,best FEV1,Sex,Date Recorded,ecFEV1,ecFEF2575,ecFEF2575%ecFEV1,Predicted FEV1,ecFEV1 % Predicted,FEV1 % Predicted,best FEV1 old,idx FEV1,idx FEF2575%FEV1,idx best FEV1
0,B155916,32,162,1.50,0.47,1.64,Female,2019-01-01,1.50,0.47,31.333333,3.158943,47.484238,47.484238,1.64,30,15,32
1,B155917,45,175,2.67,1.00,2.67,Male,2019-01-01,2.67,1.00,37.453182,3.937246,67.813909,67.813909,2.67,53,18,53
2,B155918,34,191,4.82,3.48,4.99,Male,2019-01-01,4.82,3.48,72.199168,5.141756,93.742289,93.742289,4.99,96,36,99
3,B155921,34,150,1.44,0.59,1.45,Female,2019-01-01,1.44,0.59,40.972219,2.653810,54.261617,54.261617,1.45,28,20,29
4,B155925,38,167,0.92,0.34,1.33,Female,2019-01-01,0.92,0.34,36.956521,3.244338,28.357098,28.357098,1.33,18,18,26


In [ ]:
df = pd.concat([df19, df23]).sort_values("ID")

In [ ]:
(df.groupby("ID").apply(lambda df: len(df)) > 1).sum()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_32318/3045146862.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  (df.groupby('ID').apply(lambda df: len(df)) > 1).sum()


1485

In [ ]:
df = bd.load_meas_from_excel("CF_Registry_19_processed_with_idx", study_folder="CFR")

INFO:root:* Checking for same day measurements *


In [ ]:
ids_with_2_entries = (
    df["ID"].value_counts()[df["ID"].value_counts() == 2].index.tolist()
)
df = df[df.ID.isin(ids_with_2_entries)]

df.to_excel(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/CF_Registry_19_23_stricly_2_entries_processed_with_idx.xlsx",
    index=False,
)

## Use 2019 / 2023 data only

In [ ]:
cols2read = [
    "s01caseid_original",
    # "s01sex",
    "s01height",
    "s01encounterageyears",
    "s03cliqtrfev1",  # Value at annual review
    "s03clibestfev1",
    "s03clifef2575",  # Value at annual review
]
colnames = ["ID", "Height", "Age", "FEV1", "best FEV1", "FEF2575"]

df = cfrd.build_cfr_df(2019, cols2read, colnames)

KeyboardInterrupt: 

In [ ]:
df["best FEV1 old"] = df["best FEV1"]
df["best FEV1"] = df["best FEV1"].where(df["best FEV1"] >= df["FEV1"], df["FEV1"])

In [ ]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed_with_idx.xlsx",
    index=False,
)

# Prep running inference

## Add obs indices

In [55]:
df = bd.load_meas_from_excel(
    "CF_Registry_15_processed_with_bFEV1_2012-15_IV_data_2015-18",
    study_folder="CFR"
)

INFO:root:* Checking for same day measurements *


In [56]:
df.head()

,ID,Sex,Height,FEV1,best FEV1,Date Recorded,Age,best FEV1 2012-15,best FEV1 year,IV days 2015,IV days 2015-16,IV days 2015-17,IV days 2015-18
0,B155916,Female,164.0,1.56,1.84,2015-01-01,28,1.99,2014-01-01,15,23.5,27.333333,30.75
1,B155917,Male,175.0,2.91,3.05,2015-01-01,42,3.47,2012-01-01,14,23.0,15.333333,19.50
2,B155918,Male,191.0,4.57,4.95,2015-01-01,30,5.20,2013-01-01,0,0.0,0.000000,0.00
3,B155920,Male,171.0,4.12,4.56,2015-01-01,35,4.82,2012-01-01,0,0.0,4.666667,3.50
4,B155921,Female,150.0,1.23,1.42,2015-01-01,30,1.51,2012-01-01,38,42.5,33.000000,35.75


In [57]:
# Add indices for model
# height = df.Height.iloc[0]
# age = df.Age.iloc[0]
# sex = df.Sex.iloc[0]
# ar_prior = "uniform"
# ecfev1_noise_model_cpt_suffix = "_std_add_mult_ecfev1"
# ar_fef2575_cpt_suffix = "_ecfev1_2_days_model_add_mult_noise"
# (
#     HFEV1,
#     uFEV1,
#     ecFEV1,
#     AR,
#     ecFEF2575prctecFEV1,
# ) = var_builders.fev1_fef2575_point_in_time_model_noise_shared_healthy_vars(
#     height,
#     age,
#     sex,
#     ar_prior,
#     ecfev1_noise_model_cpt_suffix,
#     ar_fef2575_cpt_suffix,
# )

import models.helpers as mh

ecFEV1 = mh.VariableNode("ecFEV1 (L)", 0, 6, 0.05, prior=None)
ecFEF2575prctecFEV1 = mh.VariableNode("ecFEF25-75 % ecFEV1 (%)", 0, 200, 2, prior=None)

# df[f"idx {ecFEV1.name}"] = df.apply(
#     lambda row: ecFEV1.get_bin_idx_for_value(row["ecFEV1"]), axis=1
# )
# df[f"idx {ecFEF2575prctecFEV1.name}"] = df.apply(
#     lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
#     axis=1,
# )

df[f"idx FEV1"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row["FEV1"]), axis=1
)
# df[f"idx FEF2575%FEV1"] = df.apply(
#     lambda row: ecFEF2575prctecFEV1.get_bin_idx_for_value(row["ecFEF2575%ecFEV1"]),
#     axis=1,
# )
# bfev1_col = "best FEV1 2016-19"
bfev1_col = "best FEV1 2012-15"
df[f"idx {bfev1_col}"] = df.apply(
    lambda row: ecFEV1.get_bin_idx_for_value(row[bfev1_col]), axis=1
)

In [139]:
df19 = df19.drop(columns=["bFEV1 diff", "bFEV1 % diff"])

In [ ]:
df19.to_csv(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_19_processed_bFEV1_2016-19.csv",
    index=False,
)

## Custom inference (for 2019 or 2023 only data)

In [4]:
df = bd.load_meas_from_excel(
    "CF_Registry_19_processed_bFEV1_2016-19",
    study_folder="CFR",
    use_csv=True,
)

INFO:root:* Checking for same day measurements *


In [ ]:
# df = bd.load_meas_from_excel("infer_all_19_data_with_best_FEV1", study_folder="CFR")

print(f"Initial shape {df.shape}")
df.dropna(subset=["FEV1", "FEF2575", "best FEV1"])
print(f"Final shape {df.shape} - Any rows dropped?")

INFO:root:* Checking for same day measurements *


Initial shape (2037, 21)
Final shape (2037, 21) - Any rows dropped?


### 1-day model

In [ ]:
dftmp = df.copy()
bfev1_col = "best FEV1 2012-15"
dftmp = cfrh.compute_hfev1_priors_1day_model(dftmp, bFEV1=bfev1_col)

In [ ]:
# AR = mh.VariableNode("Airway resistance (%)", 0, 90, 2, prior={"type": "uniform"})
# dftmp[AR.name] = dftmp.apply(cfrh.run_ve, axis=1)

In [ ]:
df = bd.load_meas_from_excel(
    "CF_Registry_15_processed_with_bFEV1_2012-15_IV_data_2015-18",
    study_folder="CFR"
)

In [85]:
HFEV1 = mh.VariableNode("Healthy FEV1 (L)", 1, 6, 0.05, prior=None)

dftmp['FEV1%PredST'] = dftmp['FEV1'] / dftmp.apply(lambda x: HFEV1.get_mean(x['P(HFEV1|FEV1)']), axis=1) * 100
dftmp['FEV1%PredFT'] = dftmp['FEV1'] / dftmp.apply(lambda x: HFEV1.get_mean(x['P(HFEV1|bFEV1)']), axis=1) * 100
dftmp['ppFEV1FT - ppFEV1ST'] = dftmp['FEV1%PredFT'] - dftmp['FEV1%PredST']

In [93]:
dftmp.to_excel(
    dh.get_path_to_main()
    + "ExcelFiles/CFR/ppfev1st_ft_bFEV1_2012-15_IV_2015-18_assoc.xlsx",
    index=False,
)

### 2-day model

In [64]:
bfev1_col = "best FEV1 2012-15"
dftmp = cfrh.compute_hfev1_priors_2day_model(dftmp, "FEV1", bfev1_col)

In [65]:
dftmp.head()

,ID,Sex,Height,FEV1,best FEV1,Date Recorded,Age,best FEV1 2012-15,best FEV1 year,IV days 2015,IV days 2015-16,IV days 2015-17,IV days 2015-18,idx FEV1,idx best FEV1 2012-15,P(HFEV1|FEV1),P(HFEV1|bFEV1),P(HFEV1|FEV1)_2day,P(HFEV1|bFEV1)_2day
0,B155916,Female,164.0,1.56,1.84,2015-01-01,28,1.99,2014-01-01,15,23.5,27.333333,30.75,31,39,"[4.0001807969980236e-48, 9.44835664708384e-41,...","[7.917975245088212e-120, 7.737950250708823e-10...","[4.0001807969980236e-48, 9.44835664708384e-41,...","[7.917975245088212e-120, 7.737950250708823e-10..."
1,B155917,Male,175.0,2.91,3.05,2015-01-01,42,3.47,2012-01-01,14,23.0,15.333333,19.50,58,69,"[0.0, 0.0, 0.0, 2.1287e-319, 2.341607810482806...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 2.1287e-319, 2.341607810482806...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,B155918,Male,191.0,4.57,4.95,2015-01-01,30,5.20,2013-01-01,0,0.0,0.000000,0.00,91,104,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,B155920,Male,171.0,4.12,4.56,2015-01-01,35,4.82,2012-01-01,0,0.0,4.666667,3.50,82,96,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,B155921,Female,150.0,1.23,1.42,2015-01-01,30,1.51,2012-01-01,38,42.5,33.000000,35.75,24,30,"[1.219863028527376e-12, 4.817714430972455e-10,...","[2.8598552473703093e-40, 1.6456272325871854e-3...","[1.219863028527376e-12, 4.817714430972455e-10,...","[2.8598552473703093e-40, 1.6456272325871854e-3..."


In [75]:
# validate that P(HFEV1|bFEV1) == P(HFEV1|bFEV1)_2day for all entries 
dftmp.apply(lambda row: row["P(HFEV1|bFEV1)"] != row["P(HFEV1|bFEV1)_2day"], axis=1).sum().sum()

0

# Load variables for associations

In [15]:
names_map = {
    "s01caseid_original": "ID",
    "s02hospivqty": "Hosp IVs",
    "s02hospivoveralltotaldays": "Hosp IV days",
    "s02homeivqty": "Home IVs",
    "s02homeivoveralltotaldays": "Home IV days",
    # I believe these are home orals because these
    # "s01coursesoforalantibiotics": "Oral",
    # Technically Non-IV hosp treatments = Orals?
    # "s02hospnonivqty": "Hosp non IVs",
    # "s02hospnonivoveralltotaldays": "Hosp non IV days",
    # complications
    # "s06acutechestepisodesqty": "Chest episodes",
    # "s06cmpscoughfractqty": "Cough episodes",
    # "s06cmpspulmabscessqty": "Pulm Abscess",
    # "s09patientsmokes": "Smoking status",
    # "s09patientsmokessecondhand": "2nd hand smoking exposure",
}
cols2read = list(names_map.keys())
colnames = list(names_map.values())

df = pd.DataFrame(columns=colnames + ["Date Recorded"])
# years = [2019, 2020, 2021, 2022, 2023]
# years = [2019, 2020]  # , 2020, 2021, 2022, 2023]
years = [2019, 2020, 2021]
for year in years:
    dftmp = cfrd.load_cfr_data(f"{year}", cols2read, colnames)
    dftmp["Date Recorded"] = datetime.date(year, 1, 1)

    df = pd.concat([df, dftmp], ignore_index=True)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_46077/2278340034.py:30: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### Process 2015-18 data

In [52]:
# Load 2015 data and apply corrections
year = 2015
df15 = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
    encoding="cp1252",
)

col2rename = {
    "Intravenous (IV) Antibiotic Days at Home": "Home IV days",
    "Intravenous (IV) Antibiotic Days in Hospital": "Hosp IV days",
    "Intravenous (IV) Antibiotic Overall Days ": "IV days",
}
df15 = df15.rename(columns=col2rename)
df15 = df15[["ID"] + list(col2rename.values())]

df15["Date Recorded"] = datetime.date(year, 1, 1)

assert (df15["IV days"] != df15["Home IV days"] + df15["Hosp IV days"]).sum() == 0
df15[df15["IV days"] > 365]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_58229/1764833477.py:3: DtypeWarning: Columns (56,59,60,69,79,80,81,87,90,91,93,94,95,96,97,99,100,101,104,113,117,118,119,124,125,131,132,133,137,142,143,148,170,193,198,217,231,232,234,243,247,248,269,279) have mixed types. Specify dtype option on import or set low_memory=False.
  df15 = pd.read_csv(


,ID,Home IV days,Hosp IV days,IV days,Date Recorded
4648,B163854,375,0,375,2015-01-01
4988,B164542,334,50,384,2015-01-01
6546,B167803,344,111,455,2015-01-01


In [53]:
year = 2016
df16 = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
    encoding="cp1252",
)

col2rename = {
    "Intravenous (IV) Antibiotic Days at Home": "Home IV days",
    "Intravenous (IV) Antibiotic Days in Hospital": "Hosp IV days",
}
df16 = df16.rename(columns=col2rename)
df16 = df16[["ID"] + list(col2rename.values())]

# ID B168895, has Hosp IV days 36537.0, should be 365
df16.loc[df16["ID"] == "B168895", "Hosp IV days"] = 365
# Value can't be negative
df16.loc[df16["ID"] == "B158183", "Home IV days"] = 346

df16["IV days"] = df16["Hosp IV days"] + df16["Home IV days"]

df16["Date Recorded"] = datetime.date(year, 1, 1)

df16[df16["IV days"] > 365]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_58229/1892385680.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df16 = pd.read_csv(


,ID,Home IV days,Hosp IV days,IV days,Date Recorded
6952,B170565,0.0,432.0,432.0,2016-01-01


In [54]:
year = 2017
df17 = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
    encoding="cp1252",
)

col2rename = {
    "Intravenous (IV) Antibiotic Days at Home": "Home IV days",
    "Intravenous (IV) Antibiotic Days in Hospital": "Hosp IV days",
}
df17 = df17.rename(columns=col2rename)
df17 = df17[["ID"] + list(col2rename.values())]

df17["IV days"] = df17["Hosp IV days"] + df17["Home IV days"]

df17["Date Recorded"] = datetime.date(year, 1, 1)

df17[df17["IV days"] > 365]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_58229/2331842073.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df17 = pd.read_csv(


,ID,Home IV days,Hosp IV days,IV days,Date Recorded
6029,B170301,322.0,46.0,368.0,2017-01-01


In [55]:
year = 2018
df18 = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF{year}.csv",
    encoding="cp1252",
)

col2rename = {
    "Intravenous (IV) Antibiotic Days at Home": "Home IV days",
    "Intravenous (IV) Antibiotic Days in Hospital": "Hosp IV days",
}
df18 = df18.rename(columns=col2rename)
df18 = df18[["ID"] + list(col2rename.values())]

df18.loc[df18["ID"] == "B168895", "Hosp IV days"] = 365

df18["IV days"] = df18["Hosp IV days"] + df18["Home IV days"]

df18["Date Recorded"] = datetime.date(year, 1, 1)

df18[df18["IV days"] > 365]

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_58229/1395446015.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df18 = pd.read_csv(


,ID,Home IV days,Hosp IV days,IV days,Date Recorded
2333,B163943,340.0,39.0,379.0,2018-01-01
4122,B160040,193.0,252.0,445.0,2018-01-01


In [57]:
df = pd.concat([df15, df16, df17, df18], ignore_index=True)
df.drop(columns=["Hosp IV days", "Home IV days"], inplace=True)
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_2015-18_IV_data.xlsx",
    index=False,
)

### Process 2018 data

In [ ]:
# Load 2018 data and apply corrections
df18 = pd.read_csv(
    dh.get_path_to_main() + f"DataFiles/CFR/CF2018.csv",
    encoding="cp1252",
)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_11401/1860549101.py:2: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  df18 = pd.read_csv(


#### Data corrections

In [ ]:
col2rename = {
    "Intravenous (IV) Antibiotic Days at Home": "Home IV days",
    "Intravenous (IV) Antibiotic Numbers at Home": "Home IVs",
    "Intravenous (IV) Antibiotic Days in Hospital": "Hosp IV days",
    "Intravenous (IV) Antibiotic Numbers in Hospital": "Hosp IVs",
}
df18 = df18.rename(columns=col2rename)
df18 = df18[["ID"] + list(col2rename.values())]

In [ ]:
# 1. Correct at Hosp IV days level
df18.loc[df18["ID"] == "B168895", "Hosp IV days"] = (
    df18.loc[df18["ID"] == "B168895", "Hosp IVs"] * 14
)

# 2. Compute IV and IV days
df18["IV days"] = df18["Home IV days"].add(df18["Hosp IV days"], fill_value=0)
df18["IVs"] = df18["Home IVs"].add(df18["Hosp IVs"], fill_value=0)

# 3. Correct at IV days level
df18.loc[df18["ID"] == "B160040", "IV days"] = 365
df18.loc[df18["ID"] == "B163943", "IV days"] = 365

#### Estimate NaN Home/Hosp IV days based on #IVs

In [ ]:
# If 0 numbers of IV episodes -> likely true 0
# If NaN number of IV episodes -> likely NaN
# If NaN number of IV days and non-NaN number of IV episodes -> take number of IV episodes * 14 days (typical duration of an IV course)

mask = df18["Home IV days"].isna() & df18["Home IVs"].notna()
df18[mask]

,ID,Home IV days,Home IVs,Hosp IV days,Hosp IVs,IV days,IVs
1350,B162236,NaN,0.0,0.0,0.0,0.0,0.0
1557,B159381,NaN,0.0,NaN,0.0,NaN,0.0
2134,B169267,NaN,0.0,0.0,0.0,0.0,0.0
3659,B167799,NaN,0.0,0.0,0.0,0.0,0.0
3811,C220445,NaN,0.0,NaN,0.0,NaN,0.0
4465,B162000,NaN,0.0,NaN,0.0,NaN,0.0
4590,C221163,NaN,0.0,NaN,0.0,NaN,0.0
7952,B170615,NaN,0.0,0.0,0.0,0.0,0.0
9380,B171505,NaN,0.0,0.0,0.0,0.0,0.0


In [ ]:
df18.loc[mask, "Home IV days"] = df18.loc[mask, "Home IVs"] * 14

In [ ]:
mask = df18["Hosp IV days"].isna() & df18["Hosp IVs"].notna()
df18[mask]

,ID,Home IV days,Home IVs,Hosp IV days,Hosp IVs,IV days,IVs
1527,B158667,14.0,1.0,NaN,0.0,14.0,1.0
1557,B159381,0.0,0.0,NaN,0.0,NaN,0.0
3811,C220445,0.0,0.0,NaN,0.0,NaN,0.0
4465,B162000,0.0,0.0,NaN,0.0,NaN,0.0
4590,C221163,0.0,0.0,NaN,0.0,NaN,0.0


In [ ]:
df18.loc[mask, "Hosp IV days"] = df18.loc[mask, "Hosp IVs"] * 14

#### Finalise processing

In [ ]:
# 4. Add date
df18["Date Recorded"] = datetime.date(2018, 1, 1)
df18 = df18[["ID", "IV days", "IVs", "Date Recorded"]]
# All NaN IVs are also NaN IV days, so we can just look at IV days
nan_iv_days_mask = df18["IV days"].isna()
print(f"{nan_iv_days_mask.sum()} rows with NaN IV days")
df18 = df18[~nan_iv_days_mask]

7 rows with NaN IV days


In [ ]:
df18.head(2)

,ID,IV days,IVs,Date Recorded
0,B161225,0.0,0.0,2018-01-01
1,B162838,32.0,3.0,2018-01-01


In [ ]:
# df18[df18['IV days'] > 360][["ID", "IV days", "Intravenous (IV) Antibiotic Days at Home", "Intravenous (IV) Antibiotic Numbers at Home", "Intravenous (IV) Antibiotic Days in Hospital", "Intravenous (IV) Antibiotic Numbers in Hospital"]]

### Process 2019 and 2020 data

#### Corrections at Hosp/Home IV days level

In [52]:
df_init = df.copy()
# df = df_init.copy()
total_days = (
    df["Home IV days"].add(df["Hosp IV days"], fill_value=0)
    # .add(df["Hosp non IV days"], fill_value=0)
)

mask = (
    (total_days.abs() > 365)
    | (df["Home IV days"].abs() > 365)
    | (df["Hosp IV days"].abs() > 365)
    # | (df["Hosp non IV days"].abs() > 365)
)
df[mask]

,ID,Hosp IVs,Hosp IV days,Home IVs,Home IV days,Date Recorded,IV days,IVs
11543,B158862,11.0,152.0,11.0,303.0,2020-01-01,455.0,22.0
12172,B160040,3.0,25.0,4.0,398.0,2020-01-01,423.0,7.0
14949,B166510,2.0,13.0,5.0,372.0,2020-01-01,385.0,7.0


In [21]:
# 2019
mask_2019 = df["Date Recorded"].astype(str) == "2019-01-01"

df.loc[(df.ID == "B157826") & mask_2019, "Hosp IV days"] = 365
df.loc[(df.ID == "B170420") & mask_2019, "Hosp IV days"] = 365

# Replace excel date serial numbers issues with estimate of 14 days per cure
df.loc[(df.ID == "C223005") & mask_2019, "Hosp IV days"] = (
    df.loc[(df.ID == "C223005") & mask_2019, "Hosp IVs"] * 14
)

# 2020
mask_2020 = df["Date Recorded"].astype(str) == "2020-01-01"

# Replace excel date serial numbers issues with estimate of 14 days per cure
# df.loc[(df.ID == "B167889") & mask_2020, "Hosp non IV days"] = (
#     df.loc[(df.ID == "B167889") & mask_2020, "Hosp non IVs"] * 14
# )
df.loc[(df.ID == "B168895") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B168895") & mask_2020, "Hosp IVs"] * 14
)
df.loc[(df.ID == "B169272") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B169272") & mask_2020, "Hosp IVs"] * 14
)
# df.loc[(df.ID == "B169726") & mask_2020, "Hosp non IV days"] = (
#     df.loc[(df.ID == "B169726") & mask_2020, "Hosp non IVs"] * 14
# )
# df.loc[(df.ID == "C223007") & mask_2020, "Hosp non IV days"] = (
#     df.loc[(df.ID == "C223007") & mask_2020, "Hosp non IVs"] * 14
# )
df.loc[(df.ID == "B168966") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B168966") & mask_2020, "Hosp IVs"] * 14
)
# df.loc[(df.ID == "B168895") & mask_2020, "Hosp non IV days"] = (
#     df.loc[(df.ID == "B168895") & mask_2020, "Hosp non IVs"] * 14
# )

# 2021
mask_2021 = df["Date Recorded"].astype(str) == "2021-01-01"

df.loc[(df.ID == "C221293") & mask_2021, "Hosp IV days"] = (
    df.loc[(df.ID == "C221293") & mask_2021, "Hosp IVs"] * 14
)

#### Estimate Nan IV days based on the #IVs

In [43]:
mask = df["Home IV days"].isna() & df["Home IVs"].notna()
# mask = (df["Home IV days"] == 0) & (df["Home IVs"] > 0)
mask = df["Hosp IV days"].isna() & df["Hosp IVs"].notna()
# mask = (df["Hosp IV days"] == 0) & (df["Hosp IVs"] > 0)
df[mask]

,ID,Hosp IVs,Hosp IV days,Home IVs,Home IV days,Date Recorded
5977,B168025,1.0,NaN,0.0,0.0,2019-01-01
10918,B157860,0.0,NaN,1.0,14.0,2020-01-01
12429,B160581,0.0,NaN,1.0,14.0,2020-01-01
23128,B162273,0.0,NaN,1.0,10.0,2021-01-01


In [41]:
# Home IVs
# No IV data for individuals
# Criteria: empty cells for many rows, including hosp or home ivs/days and fev1 during annual review
df.drop(index=df.loc[(df.ID == "C223013") & mask_2020].index, inplace=True)

df.drop(index=df.loc[(df.ID == "B161703") & mask_2021].index, inplace=True)
df.drop(index=df.loc[(df.ID == "B162002") & mask_2021].index, inplace=True)
df.drop(index=df.loc[(df.ID == "C221719") & mask_2021].index, inplace=True)

In [47]:
# Hosp IVs
df.loc[(df.ID == "B168025") & mask_2019, "Hosp IV days"] = (
    df.loc[(df.ID == "B168025") & mask_2019, "Hosp IVs"] * 14
)

df.loc[(df.ID == "B157860") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B157860") & mask_2020, "Hosp IVs"] * 14
)
df.loc[(df.ID == "B160581") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B160581") & mask_2020, "Hosp IVs"] * 14
)
df.loc[(df.ID == "B168966") & mask_2020, "Hosp IV days"] = (
    df.loc[(df.ID == "B168966") & mask_2020, "Hosp IVs"] * 14
)

df.loc[(df.ID == "B162273") & mask_2021, "Hosp IV days"] = (
    df.loc[(df.ID == "B162273") & mask_2021, "Hosp IVs"] * 14
)

#### Merging IV columns

In [49]:
# Merge home and hosp IV and IV days
df["IV days"] = df["Home IV days"].add(df["Hosp IV days"], fill_value=0)
df["IVs"] = df["Home IVs"].add(df["Hosp IVs"], fill_value=0)

# If both are NaN -> gives NaN
# If one is non NaN and the other is NaN -> gives the non-NaN value

In [51]:
# Ensure uniqueness on ID and Date Recorded
if df.duplicated(subset=["ID", "Date Recorded"]).sum() != 0:
    raise ValueError

print("Number of NaN in 'IVs':", df["IVs"].isna().sum())
# print("Number of NaN in 'Hosp non IVs':", df["Hosp non IVs"].isna().sum())
# print("Number of NaN in 'Oral':", df["Oral"].isna().sum())

df.describe().loc[["count", "mean", "std", "min", "max"]]

Number of NaN in 'IVs': 17


,Hosp IVs,Hosp IV days,Home IVs,Home IV days,IV days,IVs
count,29963.000000,29963.000000,29963.000000,29963.000000,29963.000000,29963.000000
mean,0.581083,6.794547,0.381504,4.779695,11.574242,0.962587
std,1.197919,17.295395,0.994613,15.057619,24.545866,1.808456
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,20.000000,365.000000,15.000000,398.000000,455.000000,22.000000


#### Correct at IV days level

In [ ]:
# 11 cures of Hosp and Home IVs. Prob did them simultaneously, will use max of both for total number of IVs and IV days
df.loc[(df.ID == "B158862") & mask_2020, "IV days"] = 303  # HAE
df.loc[(df.ID == "B158862") & mask_2020, "IVs"] = 11

# All year under treatment - although why 5 cures and not 1?
df.loc[(df.ID == "B160040") & mask_2020, "IV days"] = 365  # yes
df.loc[(df.ID == "B166510") & mask_2020, "IV days"] = 365

In [58]:
total_days = (
    df["Home IV days"].add(df["Hosp IV days"], fill_value=0)
    # .add(df["Hosp non IV days"], fill_value=0)
)

mask = (
    (total_days.abs() > 365)
    | (df["Home IV days"].abs() > 365)
    | (df["Hosp IV days"].abs() > 365)
    # | (df["Hosp non IV days"].abs() > 365)
)
df[mask]

,ID,Hosp IVs,Hosp IV days,Home IVs,Home IV days,Date Recorded,IV days,IVs
11543,B158862,11.0,152.0,11.0,303.0,2020-01-01,303.0,11.0
12172,B160040,3.0,25.0,4.0,398.0,2020-01-01,365.0,7.0
14949,B166510,2.0,13.0,5.0,372.0,2020-01-01,365.0,7.0


#### Antibiotics episodes and number of days should be non null together

In [ ]:
# Check cases where Oral XOR Oral days are zero
xor_mask = (df["Hosp non IVs"] == 0) ^ (df["Hosp non IV days"] == 0)
df[xor_mask][["ID", "Hosp non IVs", "Hosp non IV days"]]

,ID,Hosp non IVs,Hosp non IV days
8072,B171434,1.0,0.0


In [ ]:
# Check cases where Oral XOR Oral days are zero
xor_mask = (df["IVs"] == 0) ^ (df["IV days"] == 0)
df[xor_mask][["ID", "IVs", "IV days"]]

,ID,IVs,IV days


#### Save processed data

In [ ]:
df.drop(
    columns=["Home IVs", "Hosp IVs", "Home IV days", "Hosp IV days"],
    inplace=True,
    errors="ignore",
)

In [59]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_2019_20_21.xlsx",
    index=False,
)

## Aggregate IV data across years

In [2]:
df_iv = bd.load_meas_from_excel(
    "CF_Registry_2015-18_IV_data", study_folder="CFR", bypass_sanity_checks=True
)
df = bd.load_meas_from_excel("CF_Registry_15_processed_with_bFEV1_2012-15", study_folder="CFR", bypass_sanity_checks=True)
print("Only keeping subset of IDs that have values in 2015")
print(f"{df_iv.ID.nunique()} unique IDs in IV data")
print(f"{df.ID.nunique()} unique IDs in 2015 data")
df_iv = df_iv[df_iv.ID.isin(df.ID.unique())]

# df1920 = bd.load_meas_from_excel("IV_data_2019_20_21", study_folder="CFR")

Only keeping subset of IDs that have values in 2015
10942 unique IDs in IV data
4024 unique IDs in 2015 data


In [47]:
# years_to_keep = [2015, 2016, 2017, 2018]
years_to_keep = [2015, 2016, 2017, 2018]
dftmp = df_iv[
    df_iv["Date Recorded"].isin([datetime.date(year, 1, 1) for year in years_to_keep])
]

def aggregate_iv_data(group):
    # Exclude groups that do not have a record for 2019
    # if not (group["Date Recorded"].isin([pd.Timestamp("2019-01-01").date()]).any()):
    #     return
    total_nans = group["IV days"].isna().sum()
    contributing_zeros = (group["IV days"] == 0).sum()
    values_list = (
        group["IV days"].apply(lambda x: "nan" if pd.isna(x) else str(int(x))).tolist()
    )
    values_string = ", ".join(values_list)

    return pd.Series(
        {
            # "IVs": group["IVs"].mean(),
            "IV days": group["IV days"].mean(),
            "count_nan": total_nans,
            "count_zeros": contributing_zeros,
            "raw_values": values_string,
        }
    )

dftmp = dftmp.groupby("ID").apply(aggregate_iv_data).reset_index()

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_80661/3132117082.py:28: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [52]:
# B163854: The actual calendar days between these two annual reviews totaled 375 days (or more).
# Problem: 375 can't be an annualized IV rate. This is an isolated year, it doesn't have a corresponding short year (e.g., 355 days) to balance it out to a true 365-day average.
# Solution: Manually cap the annualized IV days at 365 to avoid skewing the annualized IV rate.

dftmp.loc[dftmp.ID == "B163854", "IV days"] = 365
dftmp.sort_values(by="IV days", ascending=False)

,ID,IV days,count_nan,count_zeros,raw_values
2528,B163854,365.00,0,0,375
1036,B158813,328.00,1,0,"328, nan"
3771,B170301,303.25,0,0,"233, 335, 368, 277"
2658,B164090,292.75,0,0,"197, 315, 338, 321"
2949,B165151,290.50,0,0,"313, 362, 205, 282"
...,...,...,...,...,...
3086,B165586,0.00,0,4,"0, 0, 0, 0"
1261,B159360,0.00,1,3,"0, nan, 0, 0"
3082,B165576,0.00,0,4,"0, 0, 0, 0"
3080,B165574,0.00,0,3,"0, 0, 0"


In [ ]:
title = "Distribution of IV days per yr in 2015 (with 2016 data added)"
fig = vh.plot_IV_days_hist(
    dftmp,
    col="IV days",
    title=title,
    save_image=False,
)
fig.update_yaxes(range=[0, 60])
fig.update_layout(width=1000, height=600)
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

In [ ]:
dftmp["Date Recorded"] = pd.to_datetime("2019-01-01").date()
dftmp.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/IV_data_2019_2020_aggregated.xlsx",
    index=False,
)

In [49]:
df = df.merge(dftmp[['ID', 'IV days']], on='ID', how='left').rename(columns={'IV days': 'IV days 2015-18'})

In [53]:
dftmp.head()

,ID,IV days,count_nan,count_zeros,raw_values
0,B155916,30.75,0,0,"15, 32, 35, 41"
1,B155917,19.50,0,1,"14, 32, 0, 32"
2,B155918,0.00,0,3,"0, 0, 0"
3,B155920,3.50,0,3,"0, 0, 14, 0"
4,B155921,35.75,0,0,"38, 47, 14, 44"


In [50]:
df.head()

,ID,Sex,Height,FEV1,best FEV1,Date Recorded,Age,best FEV1 2012-15,best FEV1 year,IV days 2015,IV days 2015-16,IV days 2015-17,IV days 2015-18
0,B155916,Female,164.0,1.56,1.84,2015-01-01,28,1.99,2014-01-01,15.0,23.5,27.333333,30.75
1,B155917,Male,175.0,2.91,3.05,2015-01-01,42,3.47,2012-01-01,14.0,23.0,15.333333,19.50
2,B155918,Male,191.0,4.57,4.95,2015-01-01,30,5.20,2013-01-01,0.0,0.0,0.000000,0.00
3,B155920,Male,171.0,4.12,4.56,2015-01-01,35,4.82,2012-01-01,0.0,0.0,4.666667,3.50
4,B155921,Female,150.0,1.23,1.42,2015-01-01,30,1.51,2012-01-01,38.0,42.5,33.000000,35.75


In [54]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/CF_Registry_15_processed_with_bFEV1_2012-15_IV_data_2015-18.xlsx",
    index=False,
)

### Plots

This is old work. I am now producing the visualisations of treatment days with the cfr viz helper functions

#### Plot number of days treatment

Note: rather plot subgroups of individuals where the model confidently disagrees

In [ ]:
# Check number of IVs, now showing normalized percentage per bar and showing the percent label per bar
import plotly.express as px

for col in ["IV days", "Hosp non IV days"]:
    fig = px.histogram(df, x=col, histnorm="percent", text_auto=".2f")
    fig.update_traces(textangle=270)
    ids = df[~df[col].isna()].ID.nunique()
    notnan = (~df[col].isna()).sum()
    title = f"{col} from 2019 registry, {notnan} entries, {ids} IDs"
    fig.update_layout(
        height=300,
        width=800,
        title=dict(text=title, font=dict(size=14)),
        margin=dict(l=40, r=20, t=40, b=40),
        xaxis_title=col,
        yaxis_title="Percent",
    )
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

# CCL: 75% of individuals had no hosp IVs between 2019 and 2023. 85% no home IVs. Trikata arrived in nov 2020

#### Plot number of IV/Oral episodes

In [ ]:
# Check number of IVs, now showing normalized percentage per bar and showing the percent label per bar
import plotly.express as px

# for col in ["Hosp IVs", "Home IVs"]:
for col in ["IVs", "Oral", "Hosp non IVs"]:
    fig = px.histogram(df, x=col, histnorm="percent", text_auto=".2f")
    fig.update_traces(textangle=270)
    ids = df[~df[col].isna()].ID.nunique()
    notnan = (~df[col].isna()).sum()
    title = f"{col} from 2019 registry, {notnan} entries, {ids} IDs"
    fig.update_layout(
        height=300,
        width=800,
        title=dict(text=title, font=dict(size=14)),
        margin=dict(l=40, r=20, t=40, b=40),
        xaxis_title=col,
        yaxis_title="Percent",
    )
    fig.show()
    # fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/{title}.pdf")

# CCL: 75% of individuals had no hosp IVs between 2019 and 2023. 85% no home IVs. Trikata arrived in nov 2020

#### Older plots

In [ ]:
# Compute avg IVs per year
# skipna is True by default meaning that nans are excluded
df_avg_ivs_per_year = (
    df.groupby("ID")
    .agg(
        {
            "Home IVs": "mean",
            "Hosp IVs": "mean",
            "Oral": "mean",
            "Any antibiotics": "mean",
        }
    )
    .rename(columns={"Home IVs": "Avg Home IVs", "Hosp IVs": "Avg Hosp IVs"})
)

df_avg_ivs_per_year.describe(percentiles=[])
# CCL: 2x more hosp IVs than home IVs
# Mean hosp IVs: 0.5 per year

,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
count,11663.000000,11663.000000,11663.000000,11663.000000
mean,0.284902,0.500503,1.750503,2.535817
std,0.688697,0.922580,1.576887,2.244520
min,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.200000,1.400000,2.000000
max,11.000000,20.000000,14.200000,25.000000


In [ ]:
df_avg_ivs_per_year

,ID,Avg Home IVs,Avg Hosp IVs,Oral,Any antibiotics
ID,,,,,
0,B155916,0.2,0.4,0.8,1.4
1,B155917,0.0,0.0,1.4,1.4
2,B155918,0.6,0.2,1.4,2.2
3,B155920,0.0,0.2,0.6,0.8
4,B155921,0.2,0.2,1.8,2.2
...,...,...,...,...,...
11658,C226145,0.0,0.0,0.0,0.0
11659,C226146,0.0,0.0,1.0,1.0
11660,C226147,0.0,2.0,7.0,9.0


In [ ]:
# Melt the dataframe to long format with columns "ID", "Avg", "Type"
df_avg_ivs_long = df_avg_ivs_per_year.melt(
    id_vars="ID",
    value_vars=["Avg Home IVs", "Avg Hosp IVs", "Oral"],
    var_name="Type",
    value_name="Avg",
)
df_avg_ivs_long

,ID,Type,Avg
0,B155916,Avg Home IVs,0.2
1,B155917,Avg Home IVs,0.0
2,B155918,Avg Home IVs,0.6
3,B155920,Avg Home IVs,0.0
4,B155921,Avg Home IVs,0.2
...,...,...,...
34984,C226145,Oral,0.0
34985,C226146,Oral,1.0
34986,C226147,Oral,7.0
34987,C226148,Oral,2.0


In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_long["Avg"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_long["Avg"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_long.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    color="Type",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    # marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 150])
fig.update_xaxes(tickangle=45)

title = f"Demography per antibiotic in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
import numpy as np

# Determine the max value for binning
max_val = df_avg_ivs_per_year["Any antibiotics"].max()
# Create bin edges: first bin for 0, then (0,1], (1,2], ..., (N-1,N]
# Ensure max_val+1 to include the rightmost data point
bin_edges = np.concatenate([np.arange(-1, np.ceil(max_val) + 1)])
bin_labels = ["0"] + [f"({i};{i+1}]" for i in range(0, int(np.ceil(max_val)))]

antibio_binned = pd.cut(
    df_avg_ivs_per_year["Any antibiotics"],
    bins=bin_edges,
    labels=bin_labels,
)

fig = px.histogram(
    df_avg_ivs_per_year.assign(antibio_binned=antibio_binned),
    x="antibio_binned",
    histnorm="percent",
    text_auto=".1f",
    category_orders={"antibio_binned": bin_labels},
)

# Make the first bar (bin for '0') white
bar_colors = ["grey"] + ["#0072b2"] * (len(bin_labels) - 1)
fig.update_traces(
    textangle=90,
    marker_color=bar_colors,
    textfont=dict(size=9),
    textposition="outside",
)
fig.update_yaxes(range=[0, 29])
fig.update_xaxes(tickangle=45)

title = f"Demography of all antibiotics in 2019-23 registry, {ids} IDs"

fig.update_layout(
    height=300,
    width=800,
    title=dict(text=title, font=dict(size=12)),
    margin=dict(l=40, r=20, t=40, b=40),
    xaxis_title="Average number of antibiotics per year (hosp/home IV or oral)",
    yaxis_title="Proportion",
)
# fig.show()
fig.write_image(f"{dh.get_path_to_main()}PlotsCFR/IV associations/{title}.pdf")

In [ ]:
df.to_excel(
    dh.get_path_to_main() + "ExcelFiles/CFR/anbitiotics_data_19-23.xlsx",
    index=False,
)

In [ ]:
year = 2023
df["Date Recorded"] = datetime.date(year, 1, 1)

In [ ]:
rename_dict = dict(zip(cols2read, colnames))
df = df.rename(columns=rename_dict)